# 04 — Ensemble Scoring and Investigation Prioritisation

This notebook shows how individual model scores are fused into a blended fraud score, how investigation priority is derived, and how alert actions are assigned.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

scored = pd.read_csv("../reports/alerts_scored.csv")
print(f"Scored alerts: {scored.shape}")
scored.head()

## Score distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ["logit_score", "hgb_score", "fraud_score", "investigation_priority_score"]):
    scored[col].hist(bins=50, ax=ax, color="steelblue", alpha=0.7)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Action and priority band distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
scored["recommended_action"].value_counts().plot.bar(ax=axes[0], color=["green", "orange", "red"])
axes[0].set_title("Recommended action")
scored["priority_band"].value_counts().plot.bar(ax=axes[1], color=["green", "orange", "red"])
axes[1].set_title("Priority band")
plt.tight_layout()
plt.show()

## Fraud score by actual label

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
scored[scored["label_fraud"]==0]["fraud_score"].hist(bins=50, alpha=0.6, label="Non-fraud", ax=ax)
scored[scored["label_fraud"]==1]["fraud_score"].hist(bins=50, alpha=0.6, label="Fraud", ax=ax)
ax.legend()
ax.set_title("Fraud score distribution by actual label")
ax.set_xlabel("Fraud score")
plt.show()

## Top alerts by investigation priority

In [ ]:
top = scored.head(20)[["transaction_id", "account_id", "amount", "fraud_score",
                        "investigation_priority_score", "recommended_action", "priority_band", "reason_codes"]]
top

## Reason code distribution

In [ ]:
all_codes = scored["reason_codes"].str.split("|").explode()
code_counts = all_codes.value_counts()
code_counts.plot.bar(color="salmon", figsize=(8, 4))
plt.title("Reason code frequency")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Key findings

- The ensemble fraud score provides clear separation between fraud and non-fraud cases.
- Investigation priority combines fraud likelihood with transaction severity.
- Reason codes provide human-readable explanations for every alert.
- The threshold settings produce a realistic distribution of approve / review / challenge actions.